[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPOSITORY/blob/main/YOUR_NOTEBOOK.ipynb)

# E/22/194

# Assignment 5: Introduction to Numerical Data Analysis


# **Foundations of Statistical Inference & Hypothesis Testing**

## Part A: Theoretical Fundamentals (Maximum Likelihood & Decision Space)

### **1.Bias of the MLE Covariance and Bessel's Correction**

The Maximum Likelihood Estimator for the covariance matrix is defined as
$$\widehat{\boldsymbol{\Sigma}}_{\text{MLE}} = \frac{1}{n} \sum_{i=1}^n (\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)(\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)^T$$.

To find its expectation,  

$$\widehat{\boldsymbol{\mu}}_n = \frac{1}{n}\sum \mathbf{X}_i$$

Because $\widehat{\boldsymbol{\mu}}_n$ is calculated from the sample itself rather than the true population mean $\boldsymbol{\mu}$, the deviations $(\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)$ are mathematically constrained to sum to zero, removing one degree of freedom.

Taking the expectation:

$$\mathbb{E}[\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}] = \mathbb{E}\left[ \frac{1}{n} \sum_{i=1}^n \left( (\mathbf{X}_i - \boldsymbol{\mu}) - (\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \right)\left( (\mathbf{X}_i - \boldsymbol{\mu}) - (\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \right)^T \right]$$


$$\mathbb{E}[\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}] = \boldsymbol{\Sigma} - \mathbb{E}\left[ (\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu})(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu})^T \right]$$

$$\mathbb{E}[\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}] = \boldsymbol{\Sigma} - \frac{1}{n}\boldsymbol{\Sigma} = \frac{n-1}{n}\boldsymbol{\Sigma}$$

This proves the raw MLE systematically underestimates the true population covariance. To resolve this, we apply Bessel's correction by multiplying the MLE by $\frac{n}{n-1}$:
$$\mathbf{S} = \frac{n}{n-1} \widehat{\boldsymbol{\Sigma}}_{\text{MLE}} \implies \mathbb{E}[\mathbf{S}] = \frac{n}{n-1} \left( \frac{n-1}{n}\boldsymbol{\Sigma} \right) = \boldsymbol{\Sigma}$$
This yields the unique, unbiased estimator $\mathbf{S}$.

### **2. Decision Space in Structural Health Monitoring**

**Type I Error ($\alpha$)**:

The probability of a False Alarm. This occurs when the diagnostic system rejects the null hypothesis ($H_0$) and flags a structural anomaly, even though the asset is completely healthy.

**Type II Error ($\beta$):**

The probability of a Missed Detection. This occurs when the system fails to reject $H_0$ and declares the asset healthy, even though a true structural fault exists.

**Mathematical Consequence of Ultra-Conservative $\alpha$:**

If an engineer sets an ultra low significance threshold  they drastically reduce the Type I error rate. However, this universally shifts the critical boundary outward, directly decreasing the statistical power ($1-\beta$) of the test. Geometrically, this inflates the hyper-volume of the healthy operation confidence ellipsoid, forcing the diagnostic system to tolerate massive deviations before triggering an alert.

## **Part B: Theoretical Extension (Asymptotic Distributions & Slutsky's Theorem)**

 **Slutsky's Theorem:**

 Let $X_n$ be a sequence of random variables that converges in distribution to a random variable $X$ ($X_n \xrightarrow{d} X$), and let $Y_n$ be a sequence of random variables that converges in probability to a constant scalar or matrix $c$ ($Y_n \xrightarrow{p} c$). Then, $X_n + Y_n \xrightarrow{d} X + c$ and $X_n Y_n \xrightarrow{d} cX$


 **Application to the CLT:**

 The Multivariate CLT proves that $\sqrt{n}(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \xrightarrow{d} \mathscr{N}(\mathbf{0}, \boldsymbol{\Sigma})$. Through the Weak Law of Large Numbers, we know the sample covariance matrix is a consistent estimator: $\mathbf{S} \xrightarrow{p} \boldsymbol{\Sigma}$. By applying Slutsky's Theorem, replacing the latent constant $\boldsymbol{\Sigma}$ with the converging sequence $\mathbf{S}$ does not alter the asymptotic distribution. This operationalizes the practical formulation:
$$\widehat{\boldsymbol{\mu}}_n \sim \mathscr{N}\left(\boldsymbol{\mu}, \frac{1}{n}\mathbf{S}\right)$$

## **Part C: Numerical Verification**

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

np.random.seed(42)
n_samples = 5000
n_features = 3

base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[[0.09, 0.02, 0.01],
         [0.02, 0.06, 0.03],
         [0.01, 0.03, 0.05]],
    size=n_samples
)

base_data[4000:, 0] += 0.015  # Sensor 1 drift
base_data[4000:, 2] -= 0.010  # Sensor 3 drift
df_strain = pd.DataFrame(base_data, columns=['Strain_Ch1', 'Strain_Ch2', 'Strain_Ch3'])
# ----------------------------------------------------------------

def verify_first_moment_homogeneity(df: pd.DataFrame, g_chunks: int = 5) -> dict:
    """
    Partitions the dataset into g chunks and evaluates first-moment homogeneity
    via Wilks' Lambda and Bartlett's Chi-Square asymptotic transformation.
    """
    data = df.values
    n, m = data.shape
    n_j = n // g_chunks

    mu_global = np.mean(data, axis=0)

    W = np.zeros((m, m))
    B = np.zeros((m, m))

    for j in range(g_chunks):
        chunk = data[j*n_j : (j+1)*n_j, :]
        mu_chunk = np.mean(chunk, axis=0)

        # Within-chunk variation
        centered_chunk = chunk - mu_chunk
        W += centered_chunk.T @ centered_chunk

        # Between-chunk variation
        mean_diff = (mu_chunk - mu_global).reshape(m, 1)
        B += n_j * (mean_diff @ mean_diff.T)

    det_W = np.linalg.det(W)
    det_T = np.linalg.det(B + W)
    wilks_lambda = det_W / det_T

    chi2_calc = -(n - 1 - (m + g_chunks) / 2) * np.log(wilks_lambda)

    df_chi2 = m * (g_chunks - 1)
    p_value = stats.chi2.sf(chi2_calc, df_chi2)

    conclusion = "Reject H0: The baseline center has drifted over time." if p_value < 0.05 else "Fail to reject H0: The first moment is homogeneous."

    return {
        "Wilks_Lambda": wilks_lambda,
        "Bartlett_Chi2": chi2_calc,
        "p_value": p_value,
        "Conclusion": conclusion
    }

results = verify_first_moment_homogeneity(df_strain, g_chunks=5)
for key, val in results.items():
    print(f"{key}: {val}")

Wilks_Lambda: 0.9904519796929591
Bartlett_Chi2: 47.9215049961488
p_value: 3.225525950915837e-06
Conclusion: Reject H0: The baseline center has drifted over time.


# **Geometric Subspace Optimization via Principal Component Analysis (PCA)**

## Part A: Theoretical Fundamentals (Coordinate Projections & Orthogonality)

**1. Diagonalization Proof**


Given $\mathbf{Z}_i = \mathbf{P}^T \widetilde{\mathbf{X}}_i$:
$$\mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T] = \mathbb{E}[(\mathbf{P}^T \widetilde{\mathbf{X}}_i)(\mathbf{P}^T \widetilde{\mathbf{X}}_i)^T] = \mathbf{P}^T \mathbb{E}[\widetilde{\mathbf{X}}_i \widetilde{\mathbf{X}}_i^T] \mathbf{P}$$
Substitute $\boldsymbol{\Sigma} = \mathbf{P}\mathbf{\Lambda}\mathbf{P}^T$:
$$\mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T] = \mathbf{P}^T (\mathbf{P}\mathbf{\Lambda}\mathbf{P}^T) \mathbf{P}$$
Because $\mathbf{P}$ is orthogonal ($\mathbf{P}^T\mathbf{P} = \mathbf{I}$):
$$\mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T] = \mathbf{I} \mathbf{\Lambda} \mathbf{I} = \mathbf{\Lambda}$$


**2. Variance Preservation**


Using the cyclic property of the trace operator:
$$\text{tr}(\boldsymbol{\Sigma}) = \text{tr}(\mathbf{P}\mathbf{\Lambda}\mathbf{P}^T) = \text{tr}(\mathbf{\Lambda}\mathbf{P}^T\mathbf{P}) = \text{tr}(\mathbf{\Lambda}\mathbf{I}) = \text{tr}(\mathbf{\Lambda}) = \sum_{j=1}^m \lambda_j$$
* **Cumulative Explained Variance Ratio:** $\Phi(k) = \frac{\sum_{j=1}^k \widehat{\lambda}_j}{\sum_{i=1}^m \widehat{\lambda}_i}$
* **Residual Unexplained Variance Ratio:** $\Psi(k) = 1.0 - \Phi(k) = \frac{\sum_{j=k+1}^m \widehat{\lambda}_j}{\sum_{i=1}^m \widehat{\lambda}_i}$

**3. Subspace Reconstruction**

Reconstructed vector:

$$\widehat{\mathbf{x}}_i = \widehat{\boldsymbol{\mu}}_n + \widehat{\mathbf{P}}_k \mathbf{z}_{i,k}$$

Residual error vector:

$$\mathbf{e}_i = \mathbf{x}_i - \widehat{\mathbf{x}}_i = \widehat{\mathbf{P}}_{m-k} \mathbf{z}_{i,m-k}$$

Pythagorean Identity:

Because the principal and residual eigenvectors are mutually orthogonal, their dot product is zero.

Thus, total energy splits perfectly:
$$\|\mathbf{x}_i - \widehat{\boldsymbol{\mu}}_n\|^2 = \|\widehat{\mathbf{P}}_k \mathbf{z}_{i,k}\|^2 + \|\widehat{\mathbf{P}}_{m-k} \mathbf{z}_{i,m-k}\|^2 = \|\mathbf{z}_{i,k}\|^2 + \|\mathbf{z}_{i,m-k}\|^2$$

 **$T^2$ vs $Q$ Diagnostics:**

 Extreme operational wind loads shift the magnitude of the existing healthy structural pathways, which strictly triggers Hotelling's $T^2$. An internal fatigue crack fundamentally alters the correlation geometry, throwing energy outside the normal eigenvectors into the unmodeled residual noise floor, immediately triggering the $Q$ statistic (SPE).

## **Part B: Numerical Verification**

In [2]:
class PCADiagnostics:
    def __init__(self, data: np.ndarray):
        self.data = data
        self.n, self.m = data.shape

    def evaluate_subspaces(self):
        mu = np.mean(self.data, axis=0)
        X_centered = self.data - mu

        S = np.cov(X_centered, rowvar=False, ddof=1)
        eigenvalues, eigenvectors = np.linalg.eigh(S)

        idx = np.argsort(eigenvalues)[::-1]
        lmbda_hat = eigenvalues[idx]
        P_hat = eigenvectors[:, idx]

        Z = X_centered @ P_hat
        results = {}

        for k in [1, 2, 3]:
            T2_array = np.sum((Z[:, :k]**2) / lmbda_hat[:k], axis=1)
            Q_array = np.sum(Z[:, k:]**2, axis=1)

            results[f"k={k}"] = {
                "Mean_T2": np.mean(T2_array),
                "Mean_Q": np.mean(Q_array)
            }
        return results

pca_engine = PCADiagnostics(df_strain.values)
print(pca_engine.evaluate_subspaces())

{'k=1': {'Mean_T2': np.float64(0.9997999999999997), 'Mean_Q': np.float64(0.09134524328398413)}, 'k=2': {'Mean_T2': np.float64(1.999599999999999), 'Mean_Q': np.float64(0.023908416626133557)}, 'k=3': {'Mean_T2': np.float64(2.9993999999999987), 'Mean_Q': np.float64(0.0)}}


#**Latent Subspace Decomposition via Factor Analysis (FA)**

## **Part A: Theoretical Fundamentals**

### **1. Fundamental Equation**
Given

$$\mathbf{Z}_i = \boldsymbol{\Lambda} \mathbf{F}_i + \boldsymbol{\epsilon}_i$$

the correlation matrix is:

$$\mathbf{R} = \mathbb{E}[(\boldsymbol{\Lambda}\mathbf{F}_i + \boldsymbol{\epsilon}_i)(\boldsymbol{\Lambda}\mathbf{F}_i + \boldsymbol{\epsilon}_i)^T]$$

$$\mathbf{R} = \boldsymbol{\Lambda}\mathbb{E}[\mathbf{F}_i\mathbf{F}_i^T]\boldsymbol{\Lambda}^T + \boldsymbol{\Lambda}\mathbb{E}[\mathbf{F}_i\boldsymbol{\epsilon}_i^T] + \mathbb{E}[\boldsymbol{\epsilon}_i\mathbf{F}_i^T]\boldsymbol{\Lambda}^T + \mathbb{E}[\boldsymbol{\epsilon}_i\boldsymbol{\epsilon}_i^T]$$

$$\mathbb{E}[\mathbf{F}_i\mathbf{F}_i^T] = \mathbf{I}$, $\mathbb{E}[\boldsymbol{\epsilon}_i\boldsymbol{\epsilon}_i^T] = \boldsymbol{\Psi}$, and $\mathbb{E}[\boldsymbol{\epsilon}_i\mathbf{F}_i^T] = \mathbf{0}$$

$$\mathbf{R} = \boldsymbol{\Lambda}\mathbf{I}\boldsymbol{\Lambda}^T + \mathbf{0} + \mathbf{0} + \boldsymbol{\Psi} = \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T + \boldsymbol{\Psi}$$

* **Communality ($h_j^2$):** $\sum_r \lambda_{j,r}^2$. The percentage of variance in sensor $j$ driven by global structural dynamics.

* **Uniqueness ($\varphi_j^2$):** The percentage of variance originating from isolated, localized instrument noise or independent channel failure.

### **2. Varimax Rotation**

* PCA eigenvectors blend multiple physical effects into single mathematical axes. Varimax rotates the axes to force loadings to be very close to $1$ or $0$, isolating distinct physical mechanisms.

* Orthogonal rotation multiplies $\boldsymbol{\Lambda}$ by an orthogonal matrix $\mathbf{T}$ ($\mathbf{T}\mathbf{T}^T = \mathbf{I}$). Thus, the approximated correlation remains strictly identical:
$$(\boldsymbol{\Lambda}\mathbf{T})(\boldsymbol{\Lambda}\mathbf{T})^T = \boldsymbol{\Lambda}\mathbf{T}\mathbf{T}^T\boldsymbol{\Lambda}^T = \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T$$

**3. Thomson’s Regression Method**

Expanding $\mathbf{Y}_i \mathbf{Y}_i^T$:
$$\mathbb{E}[\mathbf{Y}_i \mathbf{Y}_i^T] = \begin{bmatrix} \mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T] & \mathbb{E}[\mathbf{Z}_i \mathbf{F}_i^T] \\ \mathbb{E}[\mathbf{F}_i \mathbf{Z}_i^T] & \mathbb{E}[\mathbf{F}_i \mathbf{F}_i^T] \end{bmatrix} = \begin{bmatrix} \mathbf{R} & \boldsymbol{\Lambda} \\ \boldsymbol{\Lambda}^T & \mathbf{I}_{k \times k} \end{bmatrix}$$

Using the conditional projection theorem

$$\mathbb{E}[\mathbf{X}_2 | \mathbf{x}_1] = \boldsymbol{\mu}_2 + \boldsymbol{\Sigma}_{21} \boldsymbol{\Sigma}_{11}^{-1} (\mathbf{x}_1 - \boldsymbol{\mu}_1)$$

$$\mathbf{f}_i = \mathbf{0} + \boldsymbol{\Lambda}^T \mathbf{R}^{-1} (\mathbf{z}_i - \mathbf{0}) \implies \mathbf{f}_i = \boldsymbol{\Lambda}^T \mathbf{R}^{-1} \mathbf{z}_i$$

## **Part B & C: UI Architecture and FA Engine**

In [3]:
import numpy as np
import pandas as pd
from typing import Optional, Sequence, Dict, Any
from sklearn.decomposition import FactorAnalysis
import plotly.graph_objects as go
from plotly.subplots import make_subplots

class DualSubspaceDiagnosticsEngine:
    def __init__(self, df: pd.DataFrame):
        self.df = df

    def compute_empirical_pca(self, columns: Optional[Sequence[str]] = None, show_plot: bool = True) -> Dict[str, Any]:
        target_cols = list(columns) if columns else self.df.select_dtypes(include=[np.number]).columns.tolist()
        if 'count' in target_cols: target_cols.remove('count')

        X = self.df[target_cols].copy().dropna().values
        n, m = X.shape

        mu_hat = np.mean(X, axis=0)
        X_centered = X - mu_hat
        S_matrix = np.cov(X, rowvar=False, ddof=1)

        eigenvalues, eigenvectors = np.linalg.eigh(S_matrix)
        idx = np.argsort(eigenvalues)[::-1]
        lambda_hat = np.clip(eigenvalues[idx], a_min=1e-15, a_max=None)
        P_hat = eigenvectors[:, idx]

        total_variance = np.sum(lambda_hat)
        explained_variance_ratio = lambda_hat / total_variance
        cumulative_variance_ratio = np.cumsum(explained_variance_ratio)
        unexplained_variance_ratio = 1.0 - cumulative_variance_ratio

        Z_scores = np.dot(X_centered, P_hat)
        S_Z = np.cov(Z_scores, rowvar=False, ddof=1)

        k_range = np.arange(1, m)
        mean_T2_vs_k, mean_Q_vs_k = [], []

        for k in k_range:
            T2_samples = np.sum((Z_scores[:, :k] ** 2) / lambda_hat[:k], axis=1)
            mean_T2_vs_k.append(np.mean(T2_samples))
            Q_samples = np.sum(Z_scores[:, k:] ** 2, axis=1)
            mean_Q_vs_k.append(np.mean(Q_samples))

        if show_plot:
            pc_labels = [f"PC {i+1}" for i in range(m)]
            k_labels = [f"k={k}" for k in k_range]

            fig = make_subplots(
                rows=2, cols=3, horizontal_spacing=0.18, vertical_spacing=0.28,
                subplot_titles=(
                    "Feature Loading Matrix |P_hat|", "Component Values (Eigenvalues λ)",
                    "Information Profile (Explained Var.)", "Residual Space (Unexplained Var.)",
                    "Mean Hotelling's T² vs Subspace Size k", "Mean Q Statistic (SPE) vs Subspace Size k"
                )
            )

            fig.add_trace(go.Heatmap(z=np.abs(P_hat), x=pc_labels, y=target_cols, colorscale='YlOrRd', colorbar=dict(x=-0.12, len=0.38, y=0.78, yanchor="middle", xanchor="right")), row=1, col=1)
            fig.add_trace(go.Bar(x=pc_labels, y=lambda_hat, name="Eigenvalue (λ_j)", marker=dict(color='#1f77b4')), row=1, col=2)
            fig.add_trace(go.Bar(x=pc_labels, y=explained_variance_ratio * 100, name="Marginal Explained", marker=dict(color='#ff7f0e')), row=1, col=3)
            fig.add_trace(go.Scatter(x=pc_labels, y=cumulative_variance_ratio * 100, mode='lines+markers', name='Cumulative Captured', line=dict(color='#d62728', dash='dash')), row=1, col=3)
            fig.add_trace(go.Bar(x=pc_labels, y=unexplained_variance_ratio * 100, name="Remaining Noise", marker=dict(color='#2ca02c')), row=2, col=1)
            fig.add_trace(go.Scatter(x=k_labels, y=mean_T2_vs_k, mode='lines+markers', name='Mean T²', line=dict(color='#9467bd')), row=2, col=2)
            fig.add_trace(go.Scatter(x=k_labels, y=mean_Q_vs_k, mode='lines+markers', name='Mean Q (SPE)', line=dict(color='#e377c2')), row=2, col=3)

            fig.update_layout(title_text="PCA Optimization Dashboard", template="plotly_white", height=750, width=1250, margin=dict(t=150, b=60, l=140, r=80))
            fig.show()

        return {}

    def compute_empirical_fa(self, k: int, columns: Optional[Sequence[str]] = None, show_plot: bool = True) -> Dict[str, Any]:
        target_cols = list(columns) if columns else self.df.select_dtypes(include=[np.number]).columns.tolist()
        if 'count' in target_cols: target_cols.remove('count')

        X = self.df[target_cols].copy().dropna().values
        n, m = X.shape

        mu_hat = np.mean(X, axis=0)
        std_hat = np.std(X, axis=0, ddof=1)
        std_hat[std_hat == 0] = 1e-15
        Z = (X - mu_hat) / std_hat

        fa = FactorAnalysis(n_components=k, rotation='varimax', random_state=42)
        fa.fit(Z)

        lambda_matrix = fa.components_.T
        uniqueness = fa.noise_variance_
        communality = np.sum(lambda_matrix**2, axis=1)
        F_scores = fa.transform(Z)

        if show_plot:
            factor_labels = [f"Factor {j+1}" for j in range(k)]
            fig = make_subplots(
                rows=2, cols=2, horizontal_spacing=0.24, vertical_spacing=0.28,
                subplot_titles=(
                    "Structural Loadings Matrix |λ_(j,r)|", "Variance Partitioning",
                    "Sensor Uniqueness Noise Floor (φ²)", "Latent Factor Scores Variance"
                )
            )

            fig.add_trace(go.Heatmap(z=np.abs(lambda_matrix), x=factor_labels, y=target_cols, colorscale='YlOrRd', colorbar=dict(x=-0.1, len=0.38, y=0.78, yanchor="middle", xanchor="right")), row=1, col=1)
            fig.add_trace(go.Bar(y=target_cols, x=communality * 100, name="Communality (h²)", orientation='h', marker=dict(color='#1f77b4')), row=1, col=2)
            fig.add_trace(go.Bar(y=target_cols, x=uniqueness * 100, name="Uniqueness (φ²)", orientation='h', marker=dict(color='#ff7f0e')), row=1, col=2)
            fig.update_layout(barmode='stack')
            fig.add_trace(go.Scatter(x=target_cols, y=uniqueness, mode='lines+markers', name='Noise Floor (φ²)', line=dict(color='#d62728', dash='dot')), row=2, col=1)

            factor_variances = np.var(F_scores, axis=0, ddof=1)
            fig.add_trace(go.Bar(x=factor_labels, y=factor_variances, name="Factor Empirical Variance", marker=dict(color='#2ca02c')), row=2, col=2)

            fig.update_layout(title_text="FA Latent Subspace Dashboard", template="plotly_white", height=750, width=1100, margin=dict(t=100, b=60, l=80, r=80))
            fig.show()

        return {}

# Executable Runner
if __name__ == "__main__":
    np.random.seed(42)
    n_samples = 2500
    f1 = np.random.normal(0, 1, n_samples)
    f2 = np.random.normal(0, 1, n_samples)

    s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.3, n_samples)
    s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)
    s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)
    s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples)

    df_asset = pd.DataFrame(np.vstack([s1, s2, s3, s4]).T, columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4'])

    engine = DualSubspaceDiagnosticsEngine(df_asset)
    engine.compute_empirical_pca(show_plot=True)
    engine.compute_empirical_fa(k=2, show_plot=True)

# **Subspace Diagnostics & Feature De-correlation**

### **Response 1: The Trap of Variance Maximization (PCA vs. FA)**

 **1. Interpretation of Uniqueness:**

 A near-100% uniqueness ($\varphi^2$) score for `Sensor_4` reveals a complete disconnect from the asset's underlying mechanical states ($f_1$ and $f_2$). The signal is entirely dominated by independent error, pointing to a hardware malfunction, localized interference, or sheer electrical noise rather than actual structural data.

 **2. PCA's Vulnerability to Scale:**

 Because PCA strictly optimizes for the highest global variance—without filtering for signal quality or correlation—the extreme noise amplitude of `Sensor_4` ($\sigma^2 \approx 2.0$) artificially dictates the geometry of the first principal component. Consequently, `PC 1` acts merely as a noise tracker, completely sidelining the genuinely correlated data from the first three sensors, which are assigned near-zero weights on that axis.

 **3. Engineering Hazards:**

 Constructing a diagnostic threshold purely on PCA in this scenario guarantees failure. The system will establish its primary monitoring envelope around a defective sensor's static. This effectively blinds the operators to legitimate, correlated mechanical anomalies developing in the actual structure, as the monitoring tools are focused on the wrong variance.





### **Response 2: Interpretability and Orthogonal Rotation**

 **1. Overcoming the PCA Hierarchy:**

 Standard PCA strictly ranks orthogonal axes by variance scale ($\lambda_1 > \lambda_2 \dots$), which inevitably scrambles multiple sensor inputs into complex, composite variables. Factor Analysis utilizes Varimax rotation to overcome this by achieving a "simple structure." By aggressively driving the cross-loadings toward either 0 or 1, it mathematically disentangles the variables, cleanly grouping `Sensor_1` and `Sensor_2` into a single physical factor, while isolating `Sensor_3` into another.

 **2. Practical Diagnostic Value:**

 A raw PCA loading matrix forces operators to decode overlapping feature combinations when an alarm sounds, making root-cause analysis difficult. Varimax rotation solves this by mapping specific hidden factors to dedicated hardware clusters. For instance, an anomaly detected in `Factor 2` instantly directs maintenance personnel specifically to `Sensor_3`, drastically accelerating the troubleshooting process and reducing downtime.





### **Response 3: Defining the Dimensionality Boundary ($k$)**

 **1. Q-Statistic Trajectory:**

 As the retained components increase from $k=1$ to $k=2$, the residual energy (SPE) plummets significantly (from approximately 2.4 down to 0.9). Beyond $k=2$, the decay rate flattens dramatically, forming a distinct plateau commonly referred to as the "elbow."

 **2. Identifying True Dimensionality:**

 This specific geometric decay confirms that the physical system is driven by exactly two latent mechanisms. The first component absorbs the massive localized noise of `Sensor_4`, while the second maps the shared operational variance of the healthy sensors. Once $k=2$ is reached, all meaningful structural correlations have been successfully modeled.

 **3. The Cost of Over-fitting ($k=3$):**

 Expanding the subspace to $k=3$ incorporates `PC 3`, an axis exclusively composed of the residual, localized noise leftover from `Sensor_3`. By defining this unstructured white noise as part of the "healthy" parametric model, the $T^2$ boundary becomes overly sensitive, inevitably triggering false positive alarms from standard, harmless baseline fluctuations.

